# Xay Dung & Chuan Hoa Bang Nguon Thu Thap: job_sources

Notebook nay tap trung xay dung bang danh muc nguon thu thap du lieu **`job_sources`** trong kien truc Silver Layer:
- Dong vai tro la bang chieu (Dimension Table) quan ly cac kenh/san tuyen dung duoc thu thap (TopCV, TopDev, VietnamWorks).
- Cung cap khoa ngoai `source_id` cho cac bang nghiep vu va quan sat du lieu: `jobs.primary_source_id`, `job_source_records.source_id`, `job_snapshots.source_id`, va `dq_results.source_id`.
- Quan ly trang thai kich hoat (`is_active`), duong dan goc (`base_url`) va ma dinh danh ngan (`source_code`).
- Kiem tra tinh toan ven rang buoc DDL PostgreSQL (khoa chinh, unique constraints).

In [1]:
import os
import json
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np

# Cau hinh hien thi pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Moi truong xu ly bang job_sources da san sang!')

Moi truong xu ly bang job_sources da san sang!


## 1. Kham Pha & Phan Tich Nguon Du Lieu Tu Tang Bronze

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)

source_counts = Counter(l.get('source') for l in bronze_listings)
total_listings = len(bronze_listings)

source_analysis = []
for src, cnt in source_counts.items():
    sample_urls = [l.get('job_url', '') for l in bronze_listings if l.get('source') == src and l.get('job_url')]
    sample_url = sample_urls[0] if sample_urls else ''
    source_analysis.append({
        'Nguon (source)': src,
        'So luong tin': cnt,
        'Ty le dong gop (%)': round(cnt / total_listings * 100, 2),
        'Vi du job_url': sample_url[:80] + '...'
    })

df_source_stat = pd.DataFrame(source_analysis).sort_values(by='So luong tin', ascending=False)
print('--- THONG KE PHAN BO NGUON THU THAP TREN DU LIEU BRONZE (2,327 TIN) ---')
print(df_source_stat.to_string(index=False))

--- THONG KE PHAN BO NGUON THU THAP TREN DU LIEU BRONZE (2,327 TIN) ---
Nguon (source)  So luong tin  Ty le dong gop (%)                                                                       Vi du job_url
         topcv          1161               49.89 https://www.topcv.vn/viec-lam/full-stack-web-engineer-intern-fresher-ai-coding-p...
  vietnamworks          1046               44.95 https://www.vietnamworks.com/senior-allegro-pcb-layout-design-engineer-179074942...
        topdev           120                5.16    https://topdev.vn/viec-lam/network-engineer-cong-ty-tnhh-airquay-vina-2137766...


## 2. Khoi Tao & Chuan Hoa Bang job_sources Theo DDL PostgreSQL

Dinh nghia danh muc nguon thu thap voi 3 nguon hien huu (TopCV, TopDev, VietnamWorks) va 1 nguon mo rong tiem nang (ITViec).

In [3]:
now_iso = datetime.now().isoformat()

source_definitions = [
    {
        'source_id': 1,
        'source_name': 'TopCV',
        'source_code': 'topcv',
        'base_url': 'https://www.topcv.vn',
        'is_active': True,
        'created_at': now_iso,
        'updated_at': now_iso
    },
    {
        'source_id': 2,
        'source_name': 'TopDev',
        'source_code': 'topdev',
        'base_url': 'https://topdev.vn',
        'is_active': True,
        'created_at': now_iso,
        'updated_at': now_iso
    },
    {
        'source_id': 3,
        'source_name': 'VietnamWorks',
        'source_code': 'vietnamworks',
        'base_url': 'https://www.vietnamworks.com',
        'is_active': True,
        'created_at': now_iso,
        'updated_at': now_iso
    },
    {
        'source_id': 4,
        'source_name': 'ITViec',
        'source_code': 'itviec',
        'base_url': 'https://itviec.com',
        'is_active': False,  # Danh dau false cho den khi module crawler san sang
        'created_at': now_iso,
        'updated_at': now_iso
    }
]

df_job_sources = pd.DataFrame(source_definitions)

print('Khoi tao bang job_sources thanh cong:')
print(f'  - Tong so nguon thu thap duoc dinh nghia: {len(df_job_sources)}')
print(f'  - So nguon dang hoat dong (is_active = True): {(df_job_sources["is_active"] == True).sum()}')
print(f'  - Tinh duy nhat cua khoa chinh source_id: {df_job_sources["source_id"].is_unique}')
print(f'  - Tinh duy nhat cua source_name: {df_job_sources["source_name"].is_unique}')
print(f'  - Tinh duy nhat cua source_code: {df_job_sources["source_code"].is_unique}')

Khoi tao bang job_sources thanh cong:
  - Tong so nguon thu thap duoc dinh nghia: 4
  - So nguon dang hoat dong (is_active = True): 3
  - Tinh duy nhat cua khoa chinh source_id: True
  - Tinh duy nhat cua source_name: True
  - Tinh duy nhat cua source_code: True


## 3. Kiem Tra Tinh Dong Bo Voi Bang jobs & Bang Danh Muc Khac

Xac thuc rang 100% gia tri `primary_source_id` trong bang `jobs` deu ton tai va hop le trong bang `job_sources`.

In [4]:
# Map nguon tu bronze ve source_id
code_to_id = dict(zip(df_job_sources['source_code'], df_job_sources['source_id']))

mapped_ids = [code_to_id.get(l.get('source')) for l in bronze_listings]
valid_fk_count = sum(1 for sid in mapped_ids if sid in df_job_sources['source_id'].values)

print('--- KIEM TRA DONG BO TOAN VEN KHOA NGOAI (FK VALIDATION) ---')
print(f'Tong so tin tuyen dung kiem tra: {len(mapped_ids):,}')
print(f'So tin co source_id hop le trong job_sources: {valid_fk_count:,} / {len(mapped_ids)} ({valid_fk_count/len(mapped_ids)*100:.1f}%)')
print(f'Trang thai: 100% khoa ngoai tham chieu hop le!')

--- KIEM TRA DONG BO TOAN VEN KHOA NGOAI (FK VALIDATION) ---
Tong so tin tuyen dung kiem tra: 2,327
So tin co source_id hop le trong job_sources: 2,327 / 2327 (100.0%)
Trang thai: 100% khoa ngoai tham chieu hop le!


## 4. Danh Gia Cau Truc Bang & Preview Toan Bo Bang job_sources

In [5]:
print('--- 1. CAU TRUC BANG job_sources (INFO) ---')
print(df_job_sources.info())

print('\n--- 2. TOAN BO DU LIEU BANG job_sources (CHUAN POSTGRESQL) ---')
print(df_job_sources[['source_id', 'source_name', 'source_code', 'base_url', 'is_active', 'created_at']].to_string(index=False))

--- 1. CAU TRUC BANG job_sources (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   source_id    4 non-null      int64
 1   source_name  4 non-null      str  
 2   source_code  4 non-null      str  
 3   base_url     4 non-null      str  
 4   is_active    4 non-null      bool 
 5   created_at   4 non-null      str  
 6   updated_at   4 non-null      str  
dtypes: bool(1), int64(1), str(5)
memory usage: 677.0 bytes
None

--- 2. TOAN BO DU LIEU BANG job_sources (CHUAN POSTGRESQL) ---
 source_id  source_name  source_code                     base_url  is_active                 created_at
         1        TopCV        topcv         https://www.topcv.vn       True 2026-10-13T11:18:15.070744
         2       TopDev       topdev            https://topdev.vn       True 2026-10-13T11:18:15.070744
         3 VietnamWorks vietnamworks https://www.vietnamworks.com 